# M3L4 E03 — Debugging con traces [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

Tener traces está muy bien, pero **de nada sirven si no los sabés leer**. En producción, los agentes fallan de formas que no levantan excepciones: dan respuestas incorrectas, tardan demasiado, o entran en loops infinitos.

Cada uno de estos problemas deja una **firma única en el trace**. Este ejercicio te enseña a reconocer esas firmas y diagnosticar la causa raíz.

| Problema | Firma en el trace | Impacto |
|---|---|---|
| **Misclassification** | El intent del span de routing no coincide con el esperado | Respuesta incorrecta para el usuario |
| **Retrieval vacío** | Un span `retrieval` devuelve `count: 0` | El agente responde "no sé" |
| **Latencia alta** | Un span tiene `duration_ms > 3000` | Mala experiencia de usuario |
| **Loop de agentes** | El mismo agente aparece 3+ veces en los spans | La consulta nunca se resuelve |
| **Error silencioso** | Un span termina sin output o con error | El sistema falla sin notificarlo |

In [ ]:
from collections import Counter

trace_misclassification = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {'expected_intent': 'finance', 'actual_intent': 'it'},
    'spans': [
        {'name': 'orchestrator-routing', 'output': {'intent': 'it'}, 'duration_ms': 140},
        {'name': 'it-agent', 'output': {'response': 'Probá reiniciar la app.'}, 'duration_ms': 900}
    ],
    'output': {'final_response': 'Probá reiniciar la app.'}
}

trace_empty_retrieval = {
    'trace_name': 'rag-support-request',
    'input': {'query': '¿Cuál es la política de licencia por maternidad?'},
    'spans': [
        {'name': 'orchestrator-routing', 'output': {'intent': 'hr'}, 'duration_ms': 110},
        {'name': 'retrieval', 'output': {'documents': [], 'count': 0}, 'duration_ms': 250},
        {'name': 'hr-agent', 'output': {'response': 'No tengo información.'}, 'duration_ms': 820}
    ],
    'output': {'final_response': 'No tengo información disponible.'}
}

trace_high_latency = {
    'trace_name': 'support-request',
    'input': {'query': '¿Cómo solicito vacaciones?'},
    'spans': [
        {'name': 'orchestrator-routing', 'output': {'intent': 'hr'}, 'duration_ms': 95},
        {'name': 'hr-agent', 'output': {'response': 'Ingresá al portal de RRHH.'}, 'duration_ms': 5800}
    ],
    'output': {'final_response': 'Ingresá al portal de RRHH.'}
}

trace_loop = {
    'trace_name': 'support-request',
    'input': {'query': 'Mi laptop está lenta y tengo un problema con mi factura'},
    'spans': [
        {'name': 'supervisor', 'output': {'next': 'it-agent'}, 'duration_ms': 130},
        {'name': 'it-agent', 'output': {'handoff': 'finance-agent'}, 'duration_ms': 720},
        {'name': 'finance-agent', 'output': {'handoff': 'it-agent'}, 'duration_ms': 680},
        {'name': 'it-agent', 'output': {'handoff': 'finance-agent'}, 'duration_ms': 710},
        {'name': 'finance-agent', 'output': {'handoff': 'it-agent'}, 'duration_ms': 690},
    ],
    'output': {'final_response': None}
}

trace_silent_error = {
    'trace_name': 'support-request',
    'input': {'query': 'Necesito firmar un NDA'},
    'spans': [
        {'name': 'orchestrator-routing', 'output': {'intent': 'legal'}, 'duration_ms': 110},
        {'name': 'legal-agent', 'output': None, 'duration_ms': 50}
    ],
    'output': {'final_response': None}
}

print('Traces de ejemplo cargados.')

## Solución — Función `diagnose_trace()`

La función recibe un trace y aplica reglas de diagnóstico en orden de prioridad:

1. **Misclassification**: si `expected_intent != actual_intent`
2. **Retrieval vacío**: si un span de retrieval devuelve 0 documentos
3. **Latencia alta**: si un span excede los 3000 ms
4. **Loop de agentes**: si un mismo agente aparece más de 2 veces
5. **Error silencioso**: si un span termina con output `None`

Cada diagnóstico devuelve:
- `problem`: nombre del problema
- `details`: qué span y qué valores lo causaron
- `suggested_fix`: qué cambiar para resolverlo

In [ ]:
def diagnose_trace(trace: dict) -> dict:
    metadata = trace.get('metadata', {})
    spans = trace.get('spans', [])

    # 1. Misclassification
    expected = metadata.get('expected_intent')
    actual = metadata.get('actual_intent')
    if expected and actual and expected != actual:
        return {
            'problem': 'misclassification',
            'details': f'Expected intent: {expected}, actual: {actual}. Span: orchestrator-routing.',
            'suggested_fix': 'Mejorar las reglas o prompt del router para incluir más variaciones del dominio.'
        }

    # 2. Retrieval vacío
    for span in spans:
        if 'retrieval' in span.get('name', ''):
            output = span.get('output') or {}
            if isinstance(output.get('count'), int) and output['count'] == 0:
                return {
                    'problem': 'empty_retrieval',
                    'details': f"Span '{span['name']}' devolvió 0 documentos.",
                    'suggested_fix': 'Revisar la knowledge base, el índice vectorial o la query de búsqueda.'
                }

    # 3. Latencia alta
    for span in spans:
        if span.get('duration_ms', 0) > 3000:
            return {
                'problem': 'high_latency',
                'details': f"Span '{span['name']}' tardó {span['duration_ms']} ms.",
                'suggested_fix': 'Optimizar el paso, usar caché, modelo más rápido o reducir el contexto.'
            }

    # 4. Loop de agentes
    names = [s.get('name', '') for s in spans]
    counts = Counter(names)
    looping = {n: c for n, c in counts.items() if c > 2}
    if looping:
        return {
            'problem': 'agent_loop',
            'details': f'Nodos repetidos: {looping}',
            'suggested_fix': 'Agregar condición de salida o máximo de iteraciones en el supervisor.'
        }

    # 5. Error silencioso
    for span in spans:
        output = span.get('output')
        if output is None:
            return {
                'problem': 'silent_error',
                'details': f"Span '{span['name']}' terminó con output None.",
                'suggested_fix': 'Agregar try/except, logging de error y respuesta fallback en ese nodo.'
            }

    return {
        'problem': 'no_issue_detected',
        'details': 'No se encontraron problemas en este trace.',
        'suggested_fix': None
    }

print('Diagnosticador listo.')

## Ejecución — Diagnóstico de cada trace

Probamos los 5 casos. Cada uno debería detectar el problema correcto.

In [ ]:
cases = [
    ('Misclassification', trace_misclassification),
    ('Retrieval vacío',   trace_empty_retrieval),
    ('Latencia alta',     trace_high_latency),
    ('Loop de agentes',  trace_loop),
    ('Error silencioso',  trace_silent_error)
]

for name, trace in cases:
    result = diagnose_trace(trace)
    print(f'--- {name} ---')
    print(f"  Problema:     {result['problem']}")
    print(f"  Detalles:     {result['details']}")
    print(f"  Fix sugerido: {result['suggested_fix']}")
    print()

## Verificación

Cada trace debe ser diagnosticado correctamente.

In [ ]:
assert diagnose_trace(trace_misclassification)['problem'] == 'misclassification'
assert diagnose_trace(trace_empty_retrieval)['problem'] == 'empty_retrieval'
assert diagnose_trace(trace_high_latency)['problem'] == 'high_latency'
assert diagnose_trace(trace_loop)['problem'] == 'agent_loop'
assert diagnose_trace(trace_silent_error)['problem'] == 'silent_error'
print('Checks E03 OK')

## [OK] Cierre — ¿Qué logramos?

| Problema | Cómo se detecta en el trace | Fix principal |
|---|---|---|
| **Misclassification** | `metadata.expected_intent != output.intent` | Mejorar reglas del router |
| **Retrieval vacío** | Span `retrieval` con `count: 0` | Revisar knowledge base o chunking |
| **Latencia alta** | Span con `duration_ms > 3000` | Cache, modelo más rápido |
| **Loop de agentes** | Mismo agente 3+ veces | Máximo de iteraciones en supervisor |
| **Error silencioso** | Span con `output: None` | Try/except + respuesta fallback |

**¿Qué sigue?** En E04 vamos a armar un **Golden Dataset** para medir objetivamente la precisión del routing y detectar estos problemas antes de llegar a producción.